# Kaggle SSH Tunnel via Cloudflare
无需绑卡，完全免费

In [ ]:
# ── 1. 安装 SSH 服务 ─────────────────────────────────────
!apt-get install -y openssh-server -q
import os
PASSWORD = 'kaggle123'
os.system(f'echo "root:{PASSWORD}" | chpasswd')
os.system('mkdir -p /var/run/sshd')
os.system('echo "PermitRootLogin yes" >> /etc/ssh/sshd_config')
os.system('echo "PasswordAuthentication yes" >> /etc/ssh/sshd_config')
os.system('/usr/sbin/sshd')
print('SSH 服务已启动')

In [ ]:
# ── 2. 下载 cloudflared ──────────────────────────────────
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print('cloudflared 下载完成')

In [ ]:
# ── 3. 启动隧道，等待地址输出 ────────────────────────────
import subprocess, re, threading

proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'tcp://localhost:22', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print('等待隧道建立（约10秒）...')
for line in proc.stdout:
    text = line.decode()
    if 'trycloudflare.com' in text:
        m = re.search(r'https://(\S+\.trycloudflare\.com)', text)
        if m:
            cf_host = m.group(1)
            print('\n' + '='*55)
            print(f'隧道地址: {cf_host}')
            print(f'SSH 密码: {PASSWORD}')
            print()
            print('【本地终端】先安装 cloudflared:')
            print('  Windows: winget install Cloudflare.cloudflared')
            print('  Mac:     brew install cloudflared')
            print()
            print('【本地终端】连接命令:')
            print(f'  ssh -o ProxyCommand="cloudflared access ssh --hostname %h" root@{cf_host}')
            print()
            print('【VS Code】~/.ssh/config 加入以下内容:')
            print(f'  Host kaggle')
            print(f'    HostName {cf_host}')
            print(f'    User root')
            print(f'    ProxyCommand cloudflared access ssh --hostname %h')
            print('  然后 Remote-SSH: Connect to Host -> kaggle')
            print('='*55)
            break
    if 'error' in text.lower() or 'ERR' in text:
        print(text, end='')

In [ ]:
# ── 4. 安装项目依赖 ──────────────────────────────────────
!pip install ceviche -q
print('依赖安装完成，可以开始训练了')

In [ ]:
# ── 5. 保持会话存活 ──────────────────────────────────────
import time
print('保持运行中，请勿关闭...')
i = 0
while True:
    time.sleep(60)
    i += 1
    print(f'[{i}min] 运行中...')